# Phase-1: Data Ingestion & Filtering

**Goal:** Load all raw data, filter to London, and save clean outputs as parquet.

**Outputs saved to:** `outputs/phase1/`
- `phase1_crimes_london.parquet`: all street crimes for Met + City of London
- `phase1_outcomes_london.parquet`: all outcomes for Met + City of London
- `phase1_stop_search_london.parquet` : all stop & searches for Met + City of London
- `phase1_footfall_monthly.parquet`: TfL footfall aggregated to monthly per station
- `phase1_temperature_monthly.parquet`: monthly mean temperature for London (Apr 2023 – Mar 2026)

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import xarray as xr
from pathlib import Path

In [ ]:
# Dataset Loading 
BASE       = Path('Dataset Path')
OUT        = BASE / 'london-final-light' / 'outputs' / 'phase1'
OUT.mkdir(parents=True, exist_ok=True)

LONDON_FORCES = {'Metropolitan Police Service', 'City of London Police'}

# Crime archive folders in chronological order
ARCHIVES = ['0423-0424', '0524-0525', '0625-0326']

# London grid slice indices (pre-computed from lat/lon mask)
LON_Y_SLICE = slice(355, 402)
LON_X_SLICE = slice(703, 762)

print('Output folder:', OUT)
print('London forces:', LONDON_FORCES)

## Section-1: Crime Data
Load street crimes, outcomes, and stop & search across all 36 months. Filter to London forces only.

In [ ]:
# PHASE 1A: Street crimes
print('Loading street crime files...')
street_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'data' / 'crimes' / archive
    files = sorted(archive_path.rglob('*-street.csv'))
    print(f'  {archive}: {len(files)} files')
    for f in files:
        try:
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = df[df['Falls within'].isin(LONDON_FORCES)]
            if len(df) > 0:
                street_chunks.append(df)
        except Exception as e:
            print(f'  [SKIP] {f.name}: {e}')

crimes = pd.concat(street_chunks, ignore_index=True)
crimes.columns = crimes.columns.str.strip()
crimes['Month'] = pd.to_datetime(crimes['Month'], format='%Y-%m')

print(f'\nShape: {crimes.shape}')
print(f'Date range: {crimes["Month"].min().strftime("%b %Y")} → {crimes["Month"].max().strftime("%b %Y")}')
print(f'Forces: {crimes["Falls within"].unique()}')
print(f'Nulls:\n{crimes.isnull().sum()}')
crimes.head(3)

In [ ]:
# PHASE 1B: Outcomes
print('Loading outcomes files...')
outcome_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'data' / 'crimes' / archive
    files = sorted(archive_path.rglob('*-outcomes.csv'))
    print(f'  {archive}: {len(files)} files')
    for f in files:
        try:
            df = pd.read_csv(f, dtype=str, low_memory=False)
            df = df[df['Falls within'].isin(LONDON_FORCES)]
            if len(df) > 0:
                outcome_chunks.append(df)
        except Exception as e:
            print(f'  [SKIP] {f.name}: {e}')

outcomes = pd.concat(outcome_chunks, ignore_index=True)
outcomes.columns = outcomes.columns.str.strip()
outcomes['Month'] = pd.to_datetime(outcomes['Month'], format='%Y-%m')

print(f'\nShape: {outcomes.shape}')
print(f'Date range: {outcomes["Month"].min().strftime("%b %Y")} → {outcomes["Month"].max().strftime("%b %Y")}')
print(f'Nulls:\n{outcomes.isnull().sum()}')
outcomes.head(3)

In [ ]:
# PHASE 1C: Stop & Search
print('Loading stop & search files...')
ss_chunks = []

for archive in ARCHIVES:
    archive_path = BASE / 'data' / 'crimes' / archive
    # Metropolitan and City of London stop & search files only
    for pattern in ['*-metropolitan-stop-and-search.csv', '*-city-of-london-stop-and-search.csv']:
        files = sorted(archive_path.rglob(pattern))
        print(f'  {archive} / {pattern.split("-")[1]}: {len(files)} files')
        for f in files:
            try:
                df = pd.read_csv(f, dtype=str, low_memory=False)
                if len(df) > 0:
                    # Tag which force this came from using the filename
                    force = 'Metropolitan Police Service' if 'metropolitan' in f.name else 'City of London Police'
                    df['Falls within'] = force
                    ss_chunks.append(df)
            except Exception as e:
                print(f'  [SKIP] {f.name}: {e}')

stop_search = pd.concat(ss_chunks, ignore_index=True)
stop_search.columns = stop_search.columns.str.strip()
stop_search['Date'] = pd.to_datetime(stop_search['Date'], utc=True, errors='coerce')
stop_search['Month'] = stop_search['Date'].dt.to_period('M').dt.to_timestamp()

# Drop rows with no coordinates (can't spatially join later)
before = len(stop_search)
stop_search = stop_search.dropna(subset=['Latitude', 'Longitude'])
stop_search['Latitude']  = stop_search['Latitude'].astype(float)
stop_search['Longitude'] = stop_search['Longitude'].astype(float)
print(f'\nDropped {before - len(stop_search)} rows with missing coordinates')

print(f'Shape: {stop_search.shape}')
print(f'Date range: {stop_search["Month"].min().strftime("%b %Y")} → {stop_search["Month"].max().strftime("%b %Y")}')
print(f'Nulls:\n{stop_search.isnull().sum()}')
stop_search.head(3)

## Section-2: TfL Footfall
Three CSVs (2023, 2024, 2025-26) with daily station-level entry/exit counts. Aggregate to monthly totals per station.

In [ ]:
# PHASE 1D: Footfall
footfall_files = [
    BASE / 'data' / 'footfall' / 'StationFootfall_2023.csv',
    BASE / 'data' / 'footfall' / 'StationFootfall_2024.csv',
    BASE / 'data' / 'footfall' / 'StationFootfall_2025_2026 .csv',
]

print('Loading footfall files...')
ff_chunks = []
for f in footfall_files:
    df = pd.read_csv(f, dtype={'TravelDate': str})
    df.columns = df.columns.str.strip()
    ff_chunks.append(df)
    print(f'  {f.name}: {df.shape}')

footfall_raw = pd.concat(ff_chunks, ignore_index=True)
footfall_raw['TravelDate'] = pd.to_datetime(footfall_raw['TravelDate'], format='%Y%m%d')
footfall_raw['Month'] = footfall_raw['TravelDate'].dt.to_period('M').dt.to_timestamp()
footfall_raw['TotalFootfall'] = footfall_raw['EntryTapCount'] + footfall_raw['ExitTapCount']

# Aggregate daily → monthly per station
footfall_monthly = (
    footfall_raw
    .groupby(['Month', 'Station'], as_index=False)
    .agg(TotalFootfall=('TotalFootfall', 'sum'))
)

# Filter to project period: Apr 2023 – Mar 2026
footfall_monthly = footfall_monthly[
    (footfall_monthly['Month'] >= '2023-04') &
    (footfall_monthly['Month'] <= '2026-03')
].copy()

print(f'\nShape (monthly): {footfall_monthly.shape}')
print(f'Date range: {footfall_monthly["Month"].min().strftime("%b %Y")} → {footfall_monthly["Month"].max().strftime("%b %Y")}')
print(f'Unique stations: {footfall_monthly["Station"].nunique()}')
print(f'Nulls: {footfall_monthly.isnull().sum().to_dict()}')
footfall_monthly.head(3)

## Section-3 Temperature (HadUK-Grid NetCDF)

Extract monthly mean temperature for London from 1km UK grid.
- London grid slice: y-index 355–401, x-index 703–761 (pre-verified via lat/lon mask)
- Mean temperature = (tasmax + tasmin) / 2, averaged over all London grid cells

In [ ]:
def extract_london_monthly(nc_file, var_name):
    """Open a NetCDF file and return a Series of monthly mean values for London."""
    ds = xr.open_dataset(nc_file)
    london = ds[var_name].isel(
        projection_y_coordinate=LON_Y_SLICE,
        projection_x_coordinate=LON_X_SLICE
    )
    # Mean over all London grid cells for each time step
    monthly_vals = london.mean(dim=['projection_y_coordinate', 'projection_x_coordinate'])
    times  = pd.to_datetime(ds.time.values)
    ds.close()
    return pd.Series(monthly_vals.values, index=times)

TEMP_DIR = BASE / 'data' / 'temp-london'

# tasmax 
print('Extracting tasmax...')
tasmax_parts = []

# Annual files
for year in ['202301-202312', '202401-202412']:
    f = TEMP_DIR / f'tasmax_hadukgrid_uk_1km_mon_{year}.nc'
    s = extract_london_monthly(f, 'tasmax')
    tasmax_parts.append(s)
    print(f'  {f.name}: {len(s)} months')

# Individual monthly files (2025-01 through 2026-03)
for ym in ['202501','202502','202503','202504','202505','202506',
           '202507','202508','202509','202510','202511','202512',
           '202601','202602','202603']:
    f = TEMP_DIR / f'tasmax_hadukgrid_uk_1km_mon_{ym}.nc'
    if f.exists():
        s = extract_london_monthly(f, 'tasmax')
        tasmax_parts.append(s)

tasmax_series = pd.concat(tasmax_parts).sort_index()
print(f'Total tasmax months: {len(tasmax_series)}')

# tasmin 
print('\nExtracting tasmin...')
tasmin_parts = []

for year in ['202301-202312', '202401-202412']:
    f = TEMP_DIR / f'tasmin_hadukgrid_uk_1km_mon_{year}.nc'
    s = extract_london_monthly(f, 'tasmin')
    tasmin_parts.append(s)
    print(f'  {f.name}: {len(s)} months')

for ym in ['202501','202502','202503','202504','202505','202506',
           '202507','202508','202509','202510','202511','202512',
           '202601','202602','202603']:
    f = TEMP_DIR / f'tasmin_hadukgrid_uk_1km_mon_{ym}.nc'
    if f.exists():
        s = extract_london_monthly(f, 'tasmin')
        tasmin_parts.append(s)

tasmin_series = pd.concat(tasmin_parts).sort_index()
print(f'Total tasmin months: {len(tasmin_series)}')

In [ ]:
# Combine into a DataFrame and filter to project period
temp_df = pd.DataFrame({
    'tasmax': tasmax_series,
    'tasmin': tasmin_series
})
temp_df.index.name = 'Month'
temp_df['avg_temperature'] = (temp_df['tasmax'] + temp_df['tasmin']) / 2
temp_df = temp_df.reset_index()
temp_df['Month'] = pd.to_datetime(temp_df['Month']).dt.to_period('M').dt.to_timestamp()

# Filter to Apr 2023 – Mar 2026
temp_df = temp_df[
    (temp_df['Month'] >= '2023-04') &
    (temp_df['Month'] <= '2026-03')
].reset_index(drop=True)

print(f'Shape: {temp_df.shape}')
print(f'Date range: {temp_df["Month"].min().strftime("%b %Y")} → {temp_df["Month"].max().strftime("%b %Y")}')
print(f'\nTemperature stats (°C):')
print(temp_df[['tasmax', 'tasmin', 'avg_temperature']].describe().round(2))
temp_df.head(6)

## Section-4: Save Outputs

In [ ]:
# Save all outputs as parquet
saves = {
    'phase1_crimes_london.parquet':      crimes,
    'phase1_outcomes_london.parquet':    outcomes,
    'phase1_stop_search_london.parquet': stop_search,
    'phase1_footfall_monthly.parquet':   footfall_monthly,
    'phase1_temperature_monthly.parquet': temp_df,
}

for filename, df in saves.items():
    path = OUT / filename
    df.to_parquet(path, index=False)
    size_mb = path.stat().st_size / 1e6
    print(f'  Saved {filename} — {df.shape[0]:,} rows x {df.shape[1]} cols ({size_mb:.1f} MB)')

print('\nPhase 1 complete.')

In [ ]:
# --- Phase 1 summary ---
print('=' * 55)
print('PHASE 1 SUMMARY')
print('=' * 55)
print(f'Street crimes (London):   {len(crimes):>10,} rows')
print(f'Outcomes (London):        {len(outcomes):>10,} rows')
print(f'Stop & search (London):   {len(stop_search):>10,} rows')
print(f'Footfall (monthly):       {len(footfall_monthly):>10,} rows  ({footfall_monthly["Station"].nunique()} stations)')
print(f'Temperature (monthly):    {len(temp_df):>10,} rows')
print('=' * 55)
print(f'Outputs saved to: {OUT}')